In [34]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 10000
eval_iter = 250
learning_rate = 3e-4
# randomly dropping 20% of neurons to not overfit during training mode
dropout = 0.2

cuda


**Block size**: How long is each sequence?   

**Batch size**: How many sequence stacked (processed) together?   
These two hyperparameters are very useful when it comes to scaling the model.

In [12]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(len(text))

232310


In [13]:
chars = sorted(set(text))
vocab_size = len(chars)
print(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


**Type of Tokenizer** 
The code below acts as a very simple character-level tokenizer. It converts each character into integer and vice-versa. This type of tokenizer has small amount of tokens because there are only so many characters, but a large amount of work to do because of all the possible combinations (words) of them. Another type of tokenizer is the word-level tokenizer. This tokenizer has a large amount of tokens (all the words in English or other language) and a relatively small amount of work. There's also sub-word level tokenizer, which is in-between a character-level and a word-level tokenizer.

In [14]:
#Tokenizer: convert strings (characters) into integers and vice-versa
string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

print(encode("hello"))
print(decode([61, 58, 65, 65, 68]))

[61, 58, 65, 65, 68]
hello


In [15]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


Seperate the data into a training set and a validation set so that we can evaluate the performance of the model on unseen data.

In [16]:
n = int(0.8*len(data))
training_set = data[:n]
val_set = data[n:]

In [24]:
def get_batch(split):
    data = training_set if split == 'train' else val_set
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs')
print(x)
print('target')
print(y)

inputs
tensor([[65, 78,  1, 54,  1, 66, 54, 67],
        [62, 67, 60, 72,  1, 68, 59,  1],
        [68, 74, 71, 67, 58, 78,  1, 68],
        [ 1, 67, 68, 62, 72, 58,  9,  1]], device='cuda:0')
target
tensor([[78,  1, 54,  1, 66, 54, 67,  1],
        [67, 60, 72,  1, 68, 59,  1, 54],
        [74, 71, 67, 58, 78,  1, 68, 59],
        [67, 68, 62, 72, 58,  9,  1, 68]], device='cuda:0')


In [35]:
# This decorator is used here to save computation resource
# We are just reporting on the losses, so it doesn't need to compute grad
@torch.no_grad()
def estimate_loss():
    out = {}
    # turn on the evaluation mode of the model (dropout disabled)
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    # turn on the training mode of the model (dropout enabled)
    model.train()
    return out

In [18]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets = None):
        logits = self.token_embedding_table(index)

        if targets == None:
            loss = None
        else:
            # B = batch size, T = sequence size (how many tokens in one sequence), C = vocab size
            B, T, C = logits.shape
            # Needs to reshape so that each column represents a single score, instead of a sequence of scores
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is the current context of shape(B, C)
        for _ in range(max_new_tokens):
            # Get the prediction
            logits, loss = self.forward(index)
            # Get the last sequence
            logits = logits[:, -1, :]
            # Get the probability distribution of the next token
            probs = F.softmax(logits, dim=-1)
            # Using the PD to get the next token
            index_next = torch.multinomial(probs, num_samples=1)
            # Concatenate the newly generated token to the original token
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


uGv6BQq_xp![5 :Yee&ZHL"?gTn*xGo[-.L﻿:
w,r,*Vkr[4Vm﻿c4X"XI&E:﻿2'_B4X_Hq:Ia*,WN2)JT UQ3T0 FZ&[c).e&4P_M?azC Mhy"ws;.4mzq6s
STRxMwv63Y.x:]
﻿FD;[(JiwYU"FNOiOFy9T Fe-M8f(*M;*)_'8:VscmHLdYl-w;XA(vLf(0k[7!-g'&4p'?p8QtnCLW:6[D.77YQqV]LZiCp
S8QMB7fK*aRt?19f;*beErEXKGP_NjX"mOC[ZYPDlmFEcDUVgn9LisaIvxWxM?*Zz5X[Tws2'M6Z2XIvLmvaHq5_z
G_GbiOb1f19J"UWfZ8(bJgd B)W':8*k16W:n&ckkL1Oilqe9Ip*OrPX*(Vj?iv1e9giYQ﻿vq6'k!!AB5 kRtx 2mcRV(.,7Pm.QqZd_69xCV]&﻿PepW[kRti7efQqe0e]B_HqR*Kbn7U(6&4X"r*kTlu(yZa7Oi*08(tMwkJS0Q8R,t91


In [38]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# Training loop (common structure)
for iter in range(max_iters):
    if iter % eval_iter == 0:
        losses = estimate_loss()
        print(f'step: {iter}, loss: {losses}')

        
    xb, yb = get_batch('train');
    logits, loss = model.forward(xb, yb);
    # zero_grad should be on only when training on large dataset that requires the model to remember previous content
    optimizer.zero_grad(set_to_none=True);
    # correct the model and store the correction in grad
    loss.backward();
    # update the parameters using the new grad
    optimizer.step();

print(loss.item())

step: 0, loss: {'train': tensor(2.4298), 'val': tensor(2.5034)}
step: 250, loss: {'train': tensor(2.4259), 'val': tensor(2.4527)}
step: 500, loss: {'train': tensor(2.4271), 'val': tensor(2.5115)}
step: 750, loss: {'train': tensor(2.4414), 'val': tensor(2.5013)}
step: 1000, loss: {'train': tensor(2.4219), 'val': tensor(2.4720)}
step: 1250, loss: {'train': tensor(2.3992), 'val': tensor(2.4415)}
step: 1500, loss: {'train': tensor(2.4371), 'val': tensor(2.4540)}
step: 1750, loss: {'train': tensor(2.3951), 'val': tensor(2.4769)}
step: 2000, loss: {'train': tensor(2.4011), 'val': tensor(2.4751)}
step: 2250, loss: {'train': tensor(2.4346), 'val': tensor(2.4783)}
step: 2500, loss: {'train': tensor(2.4200), 'val': tensor(2.5052)}
step: 2750, loss: {'train': tensor(2.4291), 'val': tensor(2.4640)}
step: 3000, loss: {'train': tensor(2.3930), 'val': tensor(2.4781)}
step: 3250, loss: {'train': tensor(2.4242), 'val': tensor(2.4883)}
step: 3500, loss: {'train': tensor(2.4372), 'val': tensor(2.4813)}
s

In [29]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


"

ble
wike ris ang Vn ive tead walyo thano t mplyes wh
NGEThe, ace ay gideitait red be,  ge hif
Zerd PI thamyemond pthand asecabesid gre te; p haig!"Aan thero a "
tr o as I he thenind lit we wheothtly L)SEYof luprtlour.]

harthouth no ny Zecouseshithedask; onof ay.

d the ave NZe st mm at fr



thend t micly arer se, ow, w wigghan tot it
wey intrse feve, wof k the Prlllondn, h t idime'te id."
" lle sapigam MSove cke torgrd DOU,"aitowhedoved at aneslt be e t s ty. t as, han pundido.   tto t aif

